# Segmentação de instância de pets com YOLOv8-seg

Trabalho Final — TEC.1053 (Tópicos Especiais em Programação) — IFPI Campus Picos.

Este notebook executa o pipeline modular do repositório: preparação e inspeção do dataset, treino, avaliação, predições e análise de casos de baixa confiança. Antes de começar, selecione uma GPU em **Ambiente de execução > Alterar tipo de ambiente de execução**.

## 1. Clonar o repositório e instalar as dependências

In [ ]:
REPO_URL = "https://github.com/<seu-usuario>/segmentacao-pet-tep.git"  # atualize antes de executar
!git clone $REPO_URL repo
%cd repo
!pip install -q -r requirements.txt

## 2. Conferir a GPU disponível

In [ ]:
import torch
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3. Preparar o dataset

Baixa o Oxford-IIIT Pet e converte os trimaps em polígonos YOLO-seg. O split oficial de teste é mantido separado.

In [ ]:
!python -m src.prepare_dataset

## 4. Validar visualmente as anotações

In [ ]:
!python -m src.validate_annotations --num-samples 6
from pathlib import Path
from IPython.display import Image, display
for path in sorted(Path("outputs/annotation_checks").glob("*.jpg")):
    display(Image(filename=str(path), width=400))

## 5. Treinar o YOLOv8n-seg

Usa pesos pré-treinados no COCO, 30 épocas, imagens de 640 px, lote 16 e early stopping com paciência de 15 épocas.

In [ ]:
!python -m src.train --epochs 30 --imgsz 640 --batch 16 --patience 15

## 6. Avaliação formal no conjunto de teste

In [ ]:
!python -m src.evaluate --weights outputs/runs/train/weights/best.pt --split test

## 7. Gerar amostras visuais

In [ ]:
!python -m src.predict --weights outputs/runs/train/weights/best.pt --num-samples 6 --conf 0.5
for path in sorted(Path("outputs/samples").glob("*.png")):
    display(Image(filename=str(path), width=400))

## 8. Analisar casos de baixa confiança

A inferência é feita em lotes para limitar o uso de memória da GPU. Imagens sem detecção recebem confiança zero e aparecem primeiro.

In [ ]:
!python -m src.analyze_errors --weights outputs/runs/train/weights/best.pt --source data/oxford_pet_yolo/images/test --batch-size 16 --num-cases 6
for path in sorted(Path("outputs/error_cases").glob("*.jpg")):
    display(Image(filename=str(path), width=400))

## 9. Baixar os resultados

In [ ]:
import shutil
from google.colab import files
shutil.make_archive("resultados_segmentacao", "zip", "outputs")
files.download("resultados_segmentacao.zip")